# Python Mastery — Module 12
# Production AI Engineering and LLMOps

This module teaches how to move an AI/LLM prototype toward a maintainable, testable, deployable, and observable service.

## Learning outcomes
- Structure an AI project into reusable modules.
- Build a small API using FastAPI concepts and validate requests/responses.
- Manage configuration and secrets safely.
- Write unit and integration tests for AI components.
- Use structured logging, metrics, timeouts, retries, and graceful fallbacks.
- Understand Docker and reproducible environments.
- Design an evaluation pipeline and regression tests for LLM/RAG applications.
- Monitor quality, latency, cost, failures, drift, and data freshness.
- Understand CI/CD, model versioning, rollout strategies, and incident response.

The notebook is mostly runnable without external services. API, Docker, and CI examples are instructional snippets; adapt them to your installed packages and deployment platform.

## 1. Prototype versus production

A prototype answers “can this work?” Production engineering asks “can this work reliably, safely, repeatedly, and at an acceptable cost?”

| Prototype habit | Production-oriented practice |
|---|---|
| One large notebook | Reusable modules with clear interfaces |
| Hard-coded paths and keys | Environment-based configuration and secret management |
| Manual spot checks | Automated tests and repeatable evaluations |
| Print statements | Structured logs and metrics |
| Unlimited retries | Timeouts, bounded retries, and fallbacks |
| Latest model by default | Explicit model/version configuration |
| One successful demo | Monitoring, incident handling, and rollback plan |

## 2. A maintainable project structure

One reasonable structure for a small AI API:

```text
ai_service/
├── app/
│   ├── main.py              # API routes and startup
│   ├── config.py            # validated settings
│   ├── schemas.py           # request/response types
│   ├── services/
│   │   ├── retrieval.py
│   │   └── generation.py
│   └── observability.py
├── tests/
│   ├── test_schemas.py
│   ├── test_retrieval.py
│   └── test_api.py
├── data/                    # avoid committing private data
├── Dockerfile
├── requirements.txt
├── .env.example             # names/placeholders only
└── README.md
```

Keep business logic separate from HTTP handling. This makes retrieval, model calls, and evaluation easier to test without running a server.

## 3. Configuration and secrets

Configuration includes environment-specific values such as model name, timeouts, feature flags, and retrieval settings. Secrets include API keys, database passwords, and signing keys.

Rules:
- never hard-code secrets in source code or notebooks;
- do not commit `.env` files containing real credentials;
- use a secret manager or platform secrets in production;
- validate required configuration at startup;
- use separate credentials and permissions for development and production;
- rotate exposed secrets and avoid logging them.

The following example reads non-sensitive configuration from environment variables and validates values. It deliberately does not print secrets.

In [ ]:
import os
from dataclasses import dataclass

@dataclass(frozen=True)
class Settings:
    model_name: str
    request_timeout_seconds: float
    retrieval_top_k: int
    environment: str

    @classmethod
    def from_env(cls):
        timeout = float(os.getenv("REQUEST_TIMEOUT_SECONDS", "20"))
        top_k = int(os.getenv("RETRIEVAL_TOP_K", "5"))
        if timeout <= 0:
            raise ValueError("REQUEST_TIMEOUT_SECONDS must be positive")
        if not 1 <= top_k <= 50:
            raise ValueError("RETRIEVAL_TOP_K must be between 1 and 50")
        return cls(
            model_name=os.getenv("MODEL_NAME", "configured-model"),
            request_timeout_seconds=timeout,
            retrieval_top_k=top_k,
            environment=os.getenv("APP_ENV", "development"),
        )

settings = Settings.from_env()
settings

## 4. Input validation and API contracts

An API contract defines accepted inputs and returned outputs. Validate inputs before sending them to a model or database.

Useful checks:
- types and required fields;
- length and size limits;
- allowed enum values;
- request authentication and authorization;
- rate limits and quotas;
- output schema and response-size limits.

Validation reduces accidental misuse, but it is not a substitute for authorization or content-specific safety checks.

In [ ]:
from dataclasses import dataclass

@dataclass
class AskRequest:
    question: str
    top_k: int = 5

    def validate(self):
        if not isinstance(self.question, str):
            raise ValueError("question must be a string")
        if not self.question.strip():
            raise ValueError("question must not be empty")
        if len(self.question) > 4000:
            raise ValueError("question is too long")
        if not isinstance(self.top_k, int) or not 1 <= self.top_k <= 20:
            raise ValueError("top_k must be an integer from 1 to 20")
        return self

AskRequest(question="How does RAG work?", top_k=3).validate()

## 5. A minimal FastAPI service (example)

FastAPI can expose a model or RAG pipeline through HTTP. Keep route handlers thin: validate the request, call a service layer, map known errors to safe HTTP responses, and emit observability signals.

Install the packages in your environment before running a real service, for example `fastapi` and an ASGI server such as `uvicorn`. The next cell is a code template and is not executed as part of this notebook.

In [ ]:
# Example file: app/main.py
#
# from fastapi import FastAPI, HTTPException
# from pydantic import BaseModel, Field
#
# app = FastAPI(title="AI Question Answering API")
#
# class AskRequest(BaseModel):
#     question: str = Field(min_length=1, max_length=4000)
#     top_k: int = Field(default=5, ge=1, le=20)
#
# class AskResponse(BaseModel):
#     answer: str
#     evidence_ids: list[str]
#     insufficient_evidence: bool
#
# @app.get("/health")
# def health():
#     return {"status": "ok"}
#
# @app.post("/ask", response_model=AskResponse)
# def ask(request: AskRequest):
#     try:
#         result = answer_service.ask(request.question, top_k=request.top_k)
#         return AskResponse(**result)
#     except KnownUpstreamError:
#         raise HTTPException(status_code=503, detail="Answer service temporarily unavailable")
#
# Run locally: uvicorn app.main:app --reload
print("FastAPI template shown as comments; install FastAPI and implement answer_service before running.")

## 6. Unit tests and integration tests

- **Unit test:** checks a small function in isolation, such as chunking or input validation.
- **Integration test:** checks components working together, such as API → retrieval → response schema.
- **End-to-end test:** checks a realistic user journey in a deployed-like environment.
- **Regression test:** ensures previously working behavior does not silently break.

For LLM systems, exact wording is often nondeterministic. Test invariants, required fields, citation validity, known-answer behavior, and quality thresholds rather than brittle exact string matches.

In [ ]:
def normalize_question(question: str) -> str:
    if not isinstance(question, str):
        raise TypeError("question must be a string")
    return " ".join(question.split())

def test_normalize_question():
    assert normalize_question("  What   is RAG?  ") == "What is RAG?"
    assert normalize_question("") == ""
    try:
        normalize_question(None)
    except TypeError:
        pass
    else:
        raise AssertionError("Expected TypeError")

test_normalize_question()
print("Unit tests passed.")

## 7. Testing an LLM/RAG pipeline

Use a small version-controlled evaluation set containing:
- representative answerable questions;
- unanswerable questions;
- questions requiring different sources;
- ambiguous or adversarial inputs;
- known prompt-injection examples;
- expected source IDs or acceptable answer criteria.

Measure retrieval and generation separately. Example checks:
- required evidence appears in top-k;
- cited source IDs exist in retrieved context;
- answer abstains on deliberately unanswerable queries;
- JSON parses and matches schema;
- no unauthorized document is returned;
- latency and token usage remain within a budget.

Do not store sensitive user content in test fixtures unless it is approved and protected.

In [ ]:
# A small, deterministic response validator suitable for regression tests.
def validate_rag_response(payload, allowed_source_ids):
    required = {"answer", "evidence_ids", "insufficient_evidence"}
    if not isinstance(payload, dict) or set(payload) != required:
        return False
    if not isinstance(payload["answer"], str):
        return False
    if not isinstance(payload["evidence_ids"], list):
        return False
    if not all(isinstance(x, str) for x in payload["evidence_ids"]):
        return False
    if not set(payload["evidence_ids"]).issubset(set(allowed_source_ids)):
        return False
    if not isinstance(payload["insufficient_evidence"], bool):
        return False
    if payload["insufficient_evidence"] and payload["evidence_ids"]:
        return False
    return True

allowed = {"policy_refund", "policy_security"}
assert validate_rag_response({
    "answer": "Use the approved process.",
    "evidence_ids": ["policy_refund"],
    "insufficient_evidence": False,
}, allowed)
assert not validate_rag_response({
    "answer": "Made-up citation",
    "evidence_ids": ["unknown_source"],
    "insufficient_evidence": False,
}, allowed)
print("RAG response validation checks passed.")

## 8. Logging: structured, useful, and privacy-aware

Logs help explain what happened during a request. Prefer structured fields such as request ID, route, model version, elapsed time, outcome, and error category.

Avoid logging:
- API keys and credentials;
- full prompts or retrieved documents by default;
- personal or sensitive data without a clear approved purpose;
- raw model outputs if they may contain confidential information.

Use correlation IDs to connect events across services. Apply retention and access controls to logs.

In [ ]:
import logging
import time
import uuid

logger = logging.getLogger("ai_service")
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

def log_request_event(route, model_version, duration_ms, outcome, request_id=None):
    # Do not pass user prompt, secrets, or raw private documents to this logger.
    event = {
        "request_id": request_id or str(uuid.uuid4()),
        "route": route,
        "model_version": model_version,
        "duration_ms": round(float(duration_ms), 2),
        "outcome": outcome,
    }
    logger.info("request_event %s", event)
    return event

log_request_event("/ask", "configured-model", 128.4, "success")

## 9. Timeouts, retries, and graceful degradation

External model APIs and vector databases can fail or slow down. Set explicit timeouts and use bounded retries for transient errors only.

Rules of thumb:
- retry only errors that are likely transient (e.g., selected timeouts or rate limits);
- use exponential backoff and jitter;
- cap attempts and total elapsed time;
- do not blindly retry non-idempotent tool actions;
- distinguish timeout, invalid request, authentication, rate-limit, and provider errors;
- provide a safe fallback or controlled 503 when the service cannot answer.

Retries can amplify load during an outage, so they need limits and monitoring.

In [ ]:
import random

def retry_delay_seconds(attempt, base=0.5, cap=8.0, rng=None):
    if attempt < 0:
        raise ValueError("attempt must be non-negative")
    rng = rng or random
    exponential = min(cap, base * (2 ** attempt))
    return exponential * rng.uniform(0.8, 1.2)  # jitter

[round(retry_delay_seconds(i, rng=random.Random(10)), 2) for i in range(5)]

## 10. Docker and reproducible environments

Containers package application code and runtime dependencies into a reproducible image. They reduce “works on my machine” differences but do not automatically solve security, data access, or runtime configuration.

Good practices:
- pin or constrain dependencies and update them deliberately;
- use a small trusted base image;
- run as a non-root user where possible;
- keep secrets out of the image and build context;
- use `.dockerignore`;
- add health checks where supported;
- scan dependencies and images;
- configure CPU, memory, concurrency, and timeouts;
- keep model weights outside the image if they make it excessively large, unless there is a reason to bundle them.

In [ ]:
# Example Dockerfile for a small FastAPI app (adapt and save as Dockerfile):
#
# FROM python:3.12-slim
# ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
# WORKDIR /app
# COPY requirements.txt .
# RUN pip install --no-cache-dir -r requirements.txt
# COPY app ./app
# RUN useradd --create-home appuser
# USER appuser
# EXPOSE 8000
# CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
#
# Keep .env, credentials, private data, .git, and notebook outputs out of the build context.
print("Dockerfile example displayed; no container image is built by this notebook.")

## 11. CI/CD and deployment stages

A typical delivery pipeline:
1. lint and format;
2. run unit tests;
3. run integration tests with mocked external services;
4. run security and dependency checks;
5. build a container image;
6. deploy to staging;
7. run smoke tests and an evaluation suite;
8. promote gradually to production;
9. monitor and roll back if health or quality degrades.

For LLM applications, a green software test suite is not enough: run a small, repeatable model-quality evaluation as part of release validation.

## 12. Model and prompt versioning

Record enough metadata to reproduce an output or investigate a regression:
- application version / Git commit;
- model provider, model ID, and version or revision where available;
- prompt-template version;
- tokenizer and embedding-model version;
- vector index version and ingestion timestamp;
- retrieval parameters and reranker version;
- evaluation dataset version;
- generation settings and relevant package versions.

Avoid assuming a provider's moving alias always points to the same underlying model.

## 13. Monitoring: reliability, quality, and cost

Monitor multiple layers.

**Service health:** request rate, error rate, latency percentiles (p50/p95/p99), timeouts, saturation.

**AI quality:** retrieval Recall@k on an evaluation set, citation validity, abstention rate, answer quality reviews, user feedback, drift.

**Cost:** input/output tokens, embedding calls, reranker calls, cache hits, cost per request.

**Data health:** ingestion failures, stale sources, missing fields, access-control changes, index refresh status.

Production monitoring should avoid exposing private prompts or documents. Use aggregate metrics and carefully governed sampling.

In [ ]:
import numpy as np
import pandas as pd

# Demonstration: calculate latency percentiles from a small sample.
latencies_ms = np.array([85, 92, 97, 101, 110, 120, 130, 155, 210, 450])
pd.Series({
    "p50_ms": np.percentile(latencies_ms, 50),
    "p95_ms": np.percentile(latencies_ms, 95),
    "p99_ms": np.percentile(latencies_ms, 99),
    "mean_ms": latencies_ms.mean(),
}).round(1)

## 14. Rollout, rollback, and incident response

Common rollout approaches:
- **Canary:** send a small share of traffic to the new version.
- **Shadow:** run the new version alongside production without showing its output to users.
- **A/B test:** compare versions under a defined experimental design.
- **Blue/green:** maintain two deployable environments and switch traffic.

Define rollback triggers before deployment, such as error-rate increase, latency regression, citation failures, or a quality score below a threshold. An incident plan should identify owners, communication paths, data preservation requirements, and post-incident review steps.

## 15. Security and privacy checklist

- Authenticate users and authorize every data/tool action.
- Enforce document-level permissions before retrieval.
- Keep secrets in managed secret storage.
- Validate user inputs and tool arguments.
- Treat model outputs and retrieved text as untrusted.
- Prevent arbitrary code or SQL execution.
- Use least-privilege service identities.
- Apply rate limits and resource limits.
- Minimize sensitive logging and define retention.
- Test prompt injection, data exfiltration, and cross-user cache leakage.
- Have a human review path for high-impact decisions.

## 16. Capstone — productionize a mini RAG service

### Scenario
Take the Module 11 document assistant and design a deployable service.

### Deliverables
1. Separate configuration, schemas, retrieval, generation, and API modules.
2. Add request/response validation and health endpoint.
3. Add unit tests and integration tests with a mocked model client.
4. Create an evaluation set for answerable and unanswerable questions.
5. Validate citations and permissions.
6. Add structured logs without private prompt contents.
7. Configure timeouts, bounded retries, and safe fallbacks.
8. Write a Dockerfile and `.dockerignore`.
9. Add a CI pipeline for tests, linting, and evaluation.
10. Define dashboards, alert thresholds, rollback steps, and an incident runbook.
11. Document model/prompt/index versions, privacy, limitations, and operational ownership.

### Acceptance criteria
- invalid requests are rejected;
- unknown citations are rejected;
- insufficient evidence produces a safe abstention;
- external service failures do not crash the process;
- tests run without real API credentials;
- no secrets are committed or logged;
- latency, quality, and cost can be measured separately.

## 17. Exercises

1. Why should business logic be separate from API route handlers?
2. What is the difference between unit, integration, and end-to-end tests?
3. Why are exact-string tests often brittle for LLM output?
4. What should be logged for an AI request, and what should usually be excluded?
5. Why should retries be bounded?
6. What belongs in an evaluation regression suite for RAG?
7. What is the difference between canary and shadow deployment?
8. Why is a model version alone insufficient to reproduce a RAG answer?
9. Name three metrics for service health and three for AI quality.
10. Why is retrieval authorization required before passing documents to a model?

### Suggested answers

1. Separation improves testability, reuse, and the ability to change API frameworks without rewriting core logic.
2. Unit tests isolate a function; integration tests check components together; end-to-end tests validate a realistic user flow.
3. Generative outputs can vary in wording even when they satisfy the same requirements; test schemas, evidence, and quality criteria.
4. Log request IDs, version, duration, outcome, and error category; avoid secrets and unnecessary sensitive prompt/document content.
5. Unlimited retries can amplify outages, increase cost, and exceed latency budgets.
6. Include representative, unanswerable, adversarial, source-sensitive, and known-answer queries; test retrieval, citations, abstention, and output schema.
7. Canary exposes a small amount of real traffic to a new version; shadow runs it in parallel without showing its output to users.
8. Prompts, retrieval settings, index contents, embeddings, source freshness, generation settings, and package versions can also change the result.
9. Service: error rate, p95 latency, timeout rate. AI quality: Recall@k, citation validity, abstention/answer quality.
10. Otherwise, the system may expose documents the current user is not allowed to access, regardless of what the model is instructed to do.

## 18. Readiness checklist

- [ ] Structure an AI service into modules with clear interfaces.
- [ ] Validate configuration and protect secrets.
- [ ] Design API request/response schemas.
- [ ] Write unit and integration tests.
- [ ] Build RAG regression tests and quality evaluations.
- [ ] Add privacy-aware structured logging.
- [ ] Use timeouts, bounded retries, and graceful fallbacks.
- [ ] Explain Docker, CI/CD, staging, and production promotion.
- [ ] Version prompts, models, embeddings, and indexes.
- [ ] Monitor latency, errors, quality, data freshness, and cost.
- [ ] Define rollback and incident-response procedures.

### Next recommended phase
**Portfolio Integration Project:** combine your Python, data processing, machine learning, deep learning, NLP, RAG, and production skills into one end-to-end application with a documented evaluation plan and deployment-ready architecture.